# MLPerf Inference — BERT-Large / SQuAD v1.1 on Google Colab

Runs the official **MLCommons MLPerf Inference** reference benchmark
(BERT-Large, SQuAD v1.1, PyTorch backend) end-to-end on a Colab GPU:
**loadgen → model/dataset download → performance (Offline) + accuracy (f1)**.

Reference full-set accuracy target: **f1 = 90.87**. On a 1,000-example subset
you should see **f1 ≈ 90.4**, which confirms the model + harness are correct.

> **Before running:** set **Runtime → Change runtime type → GPU (T4 is fine)**.

_Adapted from a local RTX 5070 Ti run. Total runtime ~5–10 min (most of it the 1.3 GB model download)._

## 1 · Verify the GPU runtime

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0),
          "| capability:", torch.cuda.get_device_capability(0))
else:
    print("!! No GPU — set Runtime -> Change runtime type -> GPU")

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


torch 2.11.0+cu128 | CUDA available: True
device: Tesla T4 | capability: (7, 5)


## 2 · Install dependencies

LoadGen ships as a PyPI wheel (`mlcommons-loadgen`) — no C++ build needed.
`torch` is preinstalled on Colab.

In [ ]:
!pip -q install mlcommons-loadgen "transformers==4.48.3"
import mlperf_loadgen, transformers
print("loadgen + transformers", transformers.__version__, "ready")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.8 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.2/472.2 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/3.1 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 117.7 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


loadgen + transformers 4.48.3 ready


## 3 · Clone the MLPerf inference repo

In [ ]:
import os
if not os.path.isdir('/content/inference'):
    !git clone --depth 1 https://github.com/mlcommons/inference.git /content/inference
%cd /content/inference/language/bert
!git -C /content/inference rev-parse --short HEAD

Cloning into '/content/inference'...


remote: Enumerating objects: 1467, done.
remote: Counting objects: 100% (1467/1467), done.


remote: Compressing objects: 100% (1120/1120), done.


remote: Total 1467 (delta 325), reused 969 (delta 233), pack-reused 0 (from 0)
Receiving objects: 100% (1467/1467), 261.35 MiB | 30.52 MiB/s, done.


Resolving deltas: 100% (325/325), done.


Updating files: 100% (1371/1371), done.


/content/inference/language/bert


da738a5


## 4 · Download the model + dataset

BERT-Large fp32 PyTorch weights (~1.3 GB, Zenodo 3733896), the SQuAD v1.1 dev
set, and the vocab. The model download is the slow part (a few minutes, silent).

In [ ]:
%%bash
set -e
MODELDIR=build/data/bert_tf_v1_1_large_fp32_384_v2
mkdir -p $MODELDIR build/logs
[ -s build/data/dev-v1.1.json ] || wget -q -O build/data/dev-v1.1.json \
    https://raw.githubusercontent.com/rajpurkar/SQuAD-explorer/master/dataset/dev-v1.1.json
[ -s $MODELDIR/vocab.txt ] || wget -q -O $MODELDIR/vocab.txt \
    "https://zenodo.org/record/3733896/files/vocab.txt?download=1"
echo "downloading model.pytorch (~1.3 GB) ..."
[ -s $MODELDIR/model.pytorch ] || wget -q -O $MODELDIR/model.pytorch \
    "https://zenodo.org/record/3733896/files/model.pytorch?download=1"
ls -lh $MODELDIR/model.pytorch build/data/dev-v1.1.json $MODELDIR/vocab.txt

downloading model.pytorch (~1.3 GB) ...
-rw-r--r-- 1 root root 1.3G Jul 18 11:12 build/data/bert_tf_v1_1_large_fp32_384_v2/model.pytorch
-rw-r--r-- 1 root root 227K Jul 18 11:08 build/data/bert_tf_v1_1_large_fp32_384_v2/vocab.txt
-rw-r--r-- 1 root root 4.7M Jul 18 11:08 build/data/dev-v1.1.json


## 5 · Self-contained tokenizer

The reference's `tokenization` module lives in a multi-GB NVIDIA submodule.
We drop in a minimal self-contained version (standard `BasicTokenizer`, only
`unicodedata`) so no submodule checkout is needed.

In [ ]:
%%writefile tokenization.py
# Minimal self-contained tokenization for the MLPerf BERT/SQuAD reference harness.
# Provides: convert_to_unicode, printable_text, whitespace_tokenize, BasicTokenizer.
# Derived from google-research/bert tokenization.py (Apache-2.0); TensorFlow and the
# vocab/Wordpiece/FullTokenizer pieces removed (the harness uses transformers.BertTokenizer
# for wordpiece and imports only these symbols from this module).
import unicodedata


def convert_to_unicode(text):
    if isinstance(text, str):
        return text
    if isinstance(text, bytes):
        return text.decode("utf-8", "ignore")
    raise ValueError("Unsupported string type: %s" % (type(text)))


def printable_text(text):
    if isinstance(text, str):
        return text
    if isinstance(text, bytes):
        return text.decode("utf-8", "ignore")
    raise ValueError("Unsupported string type: %s" % (type(text)))


def whitespace_tokenize(text):
    text = text.strip()
    if not text:
        return []
    return text.split()


class BasicTokenizer(object):
    """Runs basic tokenization (punctuation splitting, lower casing, etc.)."""

    def __init__(self, do_lower_case=True):
        self.do_lower_case = do_lower_case

    def tokenize(self, text):
        text = convert_to_unicode(text)
        text = self._clean_text(text)
        text = self._tokenize_chinese_chars(text)
        orig_tokens = whitespace_tokenize(text)
        split_tokens = []
        for token in orig_tokens:
            if self.do_lower_case:
                token = token.lower()
                token = self._run_strip_accents(token)
            split_tokens.extend(self._run_split_on_punc(token))
        return whitespace_tokenize(" ".join(split_tokens))

    def _run_strip_accents(self, text):
        text = unicodedata.normalize("NFD", text)
        output = [c for c in text if unicodedata.category(c) != "Mn"]
        return "".join(output)

    def _run_split_on_punc(self, text):
        chars = list(text)
        i, start_new_word, output = 0, True, []
        while i < len(chars):
            char = chars[i]
            if _is_punctuation(char):
                output.append([char])
                start_new_word = True
            else:
                if start_new_word:
                    output.append([])
                start_new_word = False
                output[-1].append(char)
            i += 1
        return ["".join(x) for x in output]

    def _tokenize_chinese_chars(self, text):
        output = []
        for char in text:
            cp = ord(char)
            if self._is_chinese_char(cp):
                output.extend([" ", char, " "])
            else:
                output.append(char)
        return "".join(output)

    def _is_chinese_char(self, cp):
        return (
            (0x4E00 <= cp <= 0x9FFF) or (0x3400 <= cp <= 0x4DBF) or
            (0x20000 <= cp <= 0x2A6DF) or (0x2A700 <= cp <= 0x2B73F) or
            (0x2B740 <= cp <= 0x2B81F) or (0x2B820 <= cp <= 0x2CEAF) or
            (0xF900 <= cp <= 0xFAFF) or (0x2F800 <= cp <= 0x2FA1F)
        )

    def _clean_text(self, text):
        output = []
        for char in text:
            cp = ord(char)
            if cp == 0 or cp == 0xFFFD or _is_control(char):
                continue
            output.append(" " if _is_whitespace(char) else char)
        return "".join(output)


def _is_whitespace(char):
    if char in (" ", "\t", "\n", "\r"):
        return True
    return unicodedata.category(char) == "Zs"


def _is_control(char):
    if char in ("\t", "\n", "\r"):
        return False
    return unicodedata.category(char) in ("Cc", "Cf")


def _is_punctuation(char):
    cp = ord(char)
    if (33 <= cp <= 47) or (58 <= cp <= 64) or (91 <= cp <= 96) or (123 <= cp <= 126):
        return True
    return unicodedata.category(char).startswith("P")

Writing tokenization.py


## 6 · (Optional) Blackwell sm_120 guard

Only needed on **NVIDIA Blackwell** GPUs (RTX 50-series), where transformers'
fused SDPA kernel raises an *illegal instruction*. **Harmless on Colab's T4 / L4 /
A100** — you can run this cell everywhere. Written as `sitecustomize.py` so the
`run.py` subprocess auto-imports it and forces the pure-math attention path.

In [ ]:
%%writefile sitecustomize.py
# OPTIONAL — only needed on NVIDIA Blackwell GPUs (RTX 50-series, sm_120), where
# transformers' fused SDPA attention kernel raises CUDA "illegal instruction".
# Harmless on Colab's T4 / L4 / A100. Auto-imported by the run.py subprocess
# (cwd is on sys.path) so it forces the pure-math SDP backend everywhere.
try:
    import torch
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)
    print("[sitecustomize] math SDP forced (flash/mem-efficient disabled)", flush=True)
except Exception as e:
    print("[sitecustomize] SDP setup skipped:", e, flush=True)

Writing sitecustomize.py


## 7 · Performance run — Offline scenario

A submission-grade run targets a 10-minute minimum duration; we cap it here for a
quick demo (`user_demo.conf`). Delete that `--user_conf` flag for a full run.

In [ ]:
%%bash
cat > user_demo.conf <<'CONF'
*.Offline.target_qps = 40.0
*.Offline.min_duration = 60000
*.Offline.min_query_count = 1
CONF
python run.py --backend=pytorch --scenario=Offline --user_conf=user_demo.conf --max_examples=1000
echo "----- PERFORMANCE SUMMARY -----"
sed -n '1,16p' build/logs/mlperf_log_summary.txt

## 8 · Accuracy run + f1 score

`run.py --accuracy` runs inference, then auto-calls `accuracy-squad.py` with wrong
default paths (a harmless non-zero exit) — the accuracy **log is written regardless**,
so we score it explicitly afterwards.

In [ ]:
%%bash
python run.py --backend=pytorch --scenario=Offline --accuracy --max_examples=1000 || true
echo "----- f1 / exact_match -----"
python accuracy-squad.py \
  --vocab_file build/data/bert_tf_v1_1_large_fp32_384_v2/vocab.txt \
  --val_data build/data/dev-v1.1.json \
  --log_file build/logs/mlperf_log_accuracy.json \
  --out_file build/logs/predictions.json \
  --max_examples 1000

## Done ✅

You just ran the MLPerf Inference BERT-Large / SQuAD reference benchmark end-to-end
on a Colab GPU — a **VALID** Offline performance result plus a real **f1** on SQuAD.

**To go further:**
- Full official accuracy: change `--max_examples 1000` to the whole set (10,833) → f1 ≈ 90.87.
- Submission-grade performance: drop `--user_conf=user_demo.conf` (uses the full 10-min duration).
- Other backends: `--backend=onnxruntime` (add the ONNX model) or `--scenario=SingleStream`.